<a href="https://colab.research.google.com/github/AITrading1995/AITrading1995/blob/main/Math.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px


In [49]:
start = datetime(2020,1,1)
end = datetime.today()
symbols = ['GOOG','NVDA','GC=F','TSLA']
df = yf.download(symbols,start,end)['Close']


/tmp/ipykernel_2889/2708386968.py:4: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  4 of 4 completed


## Vector เพื่อใส่น้ำหนัก

In [50]:
W = [0.2,0.4,0.1,0.3]
vector = np.array(W)

## คำนวน ผลตอบแทน ของ สินทรัพย์
แล้วใส่น้ำหนักลงไปใน Protfolio

In [51]:
returns = np.log(df/df.shift(1))

In [52]:
volatility = returns.rolling(20).std()*np.sqrt(252)

In [53]:
volatility_melted = volatility.reset_index().melt(id_vars=['Date'], var_name='Ticker', value_name='Volatility')
plt.figure(figsize=(15, 6))
fig = px.line(volatility_melted.dropna(), x='Date', y='Volatility', color='Ticker', title='Volatility of Assets')
fig.update_layout(xaxis_title='Date', yaxis_title='Volatility')
fig.show()

<Figure size 1500x600 with 0 Axes>

In [54]:
df_melted = returns.cumsum().reset_index().melt(id_vars=['Date'], var_name='Ticker', value_name='Cumulative Return')
fig = px.line(df_melted, x='Date', y='Cumulative Return', color='Ticker', title='Cumulative Returns')
fig.update_layout(xaxis_title='Date', yaxis_title='Cumulative Return')
fig.show()

In [55]:
returns_melted = returns.reset_index().melt(id_vars=['Date'], var_name='Ticker', value_name='Daily Return')
fig = px.histogram(returns_melted.dropna(), x='Daily Return', color='Ticker', facet_col='Ticker', facet_col_wrap=2, title='Distribution of Daily Returns')
fig.update_layout(bargap=0.1) # Add some gap between bars for better visibility
fig.show()

##**คำอธิบายของ Skewness และ Kurtosis**
** **
Skewness (ความเบ้) ใช้ในการวัดความไม่สมมาตรของการกระจายความน่าจะเป็นของตัวแปรสุ่มแบบค่าจริงเมื่อเทียบกับค่าเฉลี่ย

ค่า Skewness ติดลบ หมายถึงหางด้านซ้ายของการแจกแจงยาวหรือหนากว่า แปลว่ามีโอกาสเกิดผลตอบแทนติดลบรุนแรงมากกว่า
ค่า Skewness เป็นบวก หมายถึงหางด้านขวายาวหรือหนากว่า แปลว่ามีโอกาสเกิดผลตอบแทนบวกที่รุนแรงมากกว่า

Kurtosis (ความโด่งของการแจกแจง) ใช้วัดลักษณะ “ความหนาของหาง” ของการกระจายความน่าจะเป็น

Kurtosis สูง หมายถึงการแจกแจงมีจุดยอดแหลมและหางหนา (มี outliers มาก) เมื่อเทียบกับการแจกแจงแบบปกติ
Kurtosis ต่ำ (หรือเรียกว่า platykurtic) หมายถึงจุดยอดแบนและหางบาง (มี outliers น้อย)

#**Skewness ของ Daily Returns**
** **

*   LisGC=F (ทองคำ) → -0.975
👉 เบ้ซ้ายชัดเจน → มีโอกาสเกิด “ขาลงแรง” มากกว่าปกติ (risk downside สูง)t item
* NVDA → 0.046
👉 ใกล้ 0 → การกระจายค่อนข้างสมมาตร (ไม่มี bias ชัดเจน)
* TSLA → -0.077
👉 ติดลบนิดเดียว → เกือบ symmetric แต่ยังมี bias ไปทาง downside เล็กน้อย
* ^GSPC (S&P 500) → -0.639
👉 เบ้ซ้าย → ตลาดโดยรวมมี tendency เกิด downside shock มากกว่า upside

In [56]:
print('Skewness of Daily Returns:')
print(returns.skew())


Skewness of Daily Returns:
Ticker
GC=F   -0.974609
GOOG   -0.161165
NVDA    0.046851
TSLA   -0.076326
dtype: float64


##**🧠 Insight (สำคัญสำหรับสายเทรด)**
** **
ตลาดส่วนใหญ่
(โดยเฉพาะ Index + Gold) → Negative Skew
→ ขึ้นช้า ลงแรง (classic market behavior)
หุ้นบางตัว
(เช่น NVDA) → Neutral Skew
→ เหมาะกับ strategy ที่ไม่ bias ฝั่งใดฝั่งหนึ่ง
ถ้าเจอ Negative Skew เยอะ
👉 ต้องระวัง “tail risk” (พวก crash, sell-off แรงๆ)
👉 Stop loss / risk management สำคัญมาก

##**Kurtosis ของ Daily Returns**
* GC=F (ทอง) → 9.90
👉 หางหนามาก (extreme moves บ่อย) = jump / spike บ่อย
* NVDA → 4.15
👉 สูงกว่า normal (≈3) → มี outliers พอสมควร (หุ้น growth + AI hype)
* TSLA → 3.45
👉 ใกล้ normal แต่ยังมี tail risk อยู่
* ^GSPC (S&P 500) → 14.78 🚨
👉 โคตรสูง → ตลาดโดยรวมมี “fat tails” หนักมาก
= ปกติดูนิ่ง แต่พอ crash → มาเป็นลูกใหญ่

##**🔥 สรุปโคตรสั้น**
** **
* ลบ = เสี่ยงขาลงแรง
* บวก = ลุ้นขาขึ้นแรง
* ใกล้ 0 = สมดุล

In [57]:
print('\nKurtosis of Daily Returns:')
print(returns.kurt())


Kurtosis of Daily Returns:
Ticker
GC=F    9.881282
GOOG    3.671682
NVDA    4.151526
TSLA    3.444977
dtype: float64


##**🧠 Insight (อันนี้สำคัญมาก)**
** **
1. ทุกตัว = Fat Tail

👉 ไม่มีตัวไหนเป็น normal distribution จริง
👉 ใช้ Gaussian assumption = พังแน่นอน

2. S&P 500 = Tail Risk Monster
Kurtosis สูงสุด (14.7)
Skewness ติดลบ (-0.63)

👉 แปลตรงๆ:

ตลาด “ดูปลอดภัย” แต่จริงๆ มีโอกาส crash แรงแบบไม่เตือน

3. Gold (GC=F) = Shock Asset
Skewness = ลบแรง
Kurtosis = สูง

👉 ลงแรง + มี spike
→ เหมาะ hedge แต่ก็ “กัดมือได้”

4. NVDA / TSLA
Kurtosis ปานกลาง
👉 ยังมี volatility แต่ “predictable กว่า index”
##**🔥 สรุปโคตรสั้น**
Kurtosis
* สูง = ตลาดมี “ระเบิดซ่อนอยู่”
* ยิ่งสูง = tail event ยิ่งโหด
* S&P 500 = ตัวพ่อ tail risk
* ทอง = spike + crash
* หุ้น = กลางๆ แต่ยังไม่ safe

In [58]:
# clean data
returns_clean = returns.dropna()

# check dimension
assert returns_clean.shape[1] == len(vector), "จำนวน weight ไม่ตรงกับจำนวน asset"

# portfolio return
portfolio_returns = returns_clean.dot(vector)

# ตั้งชื่อ (สำคัญสำหรับ plot)
portfolio_returns.name = "Portfolio"

# cumulative return
cumulative_portfolio_returns = (1 + portfolio_returns).cumprod() - 1


# plot
fig_portfolio = px.line(
    cumulative_portfolio_returns.reset_index(),
    x='Date',
    y='Portfolio',
    title='Cumulative Portfolio Returns'
)

fig_portfolio.update_layout(
    xaxis_title='Date',
    yaxis_title='Cumulative Return'
)

fig_portfolio.show()

In [59]:
# download data
sp500 = yf.download("^GSPC", start=start, end=end)['Close']

# ใช้ Adjusted Close (สำคัญ)
#sp500 = sp500[['Adj Close']].rename(columns={'Adj Close': '^GSPC'})

# คำนวณ return
sp500_returns = sp500.pct_change().dropna()
sp500_returns.name = "Benchmark"

/tmp/ipykernel_2889/3592480732.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


In [60]:
# align กับ portfolio
portfolio_returns, sp500_returns = portfolio_returns.align(sp500_returns.squeeze(), join='inner')

# cumulative
cum_port = (1 + portfolio_returns).cumprod() - 1
cum_bench = (1 + sp500_returns).cumprod() - 1

# รวม
compare = pd.concat([cum_port, cum_bench], axis=1)
compare.columns = ["Portfolio", "Benchmark"]

In [61]:
# align กับ portfolio
portfolio_returns, sp500_returns = portfolio_returns.align(sp500_returns.squeeze(), join='inner')

# cumulative
cum_port = (1 + portfolio_returns).cumprod() - 1
cum_bench = (1 + sp500_returns).cumprod() - 1

# รวม
compare = pd.concat([cum_port, cum_bench], axis=1)
compare.columns = ["Portfolio", "Benchmark"]

In [62]:
compare = compare.reset_index()
compare.rename(columns={'index': 'Date'}, inplace=True)

fig = px.line(compare, x='Date', y=['Portfolio', 'Benchmark'],
              title='Portfolio vs S&P 500 (^GSPC)')
fig.show()

In [80]:
vol_port = compare[['Portfolio', 'Benchmark']].rolling(20).std() * np.sqrt(252)

In [83]:
vol_port = vol_port.reset_index()
vol_port.rename(columns={'index': 'Date'}, inplace=True)

In [84]:
fig = px.line(vol_port.reset_index(), x='Date', y=['Portfolio', 'Benchmark'],
              title='Portfolio vs S&P 500 (^GSPC)')
fig.show()

In [79]:
vol_port

,Date,Portfolio,Benchmark
0,0,NaN,NaN
1,1,NaN,NaN
2,2,NaN,NaN
3,3,NaN,NaN
4,4,NaN,NaN
...,...,...,...
1575,1575,2.734517,0.916210
1576,1576,2.970112,1.017190
1577,1577,3.448596,1.136496
1578,1578,3.719825,1.205443
